Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error


Cell 2 — Load Data

In [2]:
DATA_PATH = "../data/clean_expenses.csv"  # notebook is in project/ml/
df = pd.read_csv(DATA_PATH)

df.head()


,date,type,category,amount,payment_method,description,day_name,week,is_weekend,signed_amount
0,2026-01-01,income,Income,1200000,transfer,Monthly salary,Thursday,1,False,1200000
1,2026-01-01,expense,Bills,85000,cash,Internet subscription,Thursday,1,False,-85000
2,2026-01-02,expense,Food,18000,cash,Grocery items,Friday,1,False,-18000
3,2026-01-02,expense,Transport,5000,cash,Taxi ride,Friday,1,False,-5000
4,2026-01-03,expense,Shopping,25000,card,House supplies,Saturday,1,True,-25000


Cell 3 — Parse Date + Check signed_amount

In [3]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).copy()

if "signed_amount" not in df.columns:
    raise ValueError("Missing column: signed_amount")

df["signed_amount"] = pd.to_numeric(df["signed_amount"], errors="coerce").fillna(0)

df[["date", "signed_amount"]].head()


,date,signed_amount
0,2026-01-01,1200000
1,2026-01-01,-85000
2,2026-01-02,-18000
3,2026-01-02,-5000
4,2026-01-03,-25000


Cell 4 — Weekly Aggregation (weekly_total)

In [4]:
# Week start = Monday (robust weekly grouping)
df["week_start"] = df["date"].dt.to_period("W-MON").dt.start_time

weekly = (
    df.groupby("week_start", as_index=False)["signed_amount"]
      .sum()
      .rename(columns={"signed_amount": "weekly_total"})
)

weekly = weekly.sort_values("week_start").reset_index(drop=True)
weekly


,week_start,weekly_total
0,2025-12-30,955000
1,2026-01-06,-187000
2,2026-01-13,-29000


Cell 5 — Feature: week_index

In [5]:
weekly["week_index"] = np.arange(len(weekly))

X = weekly[["week_index"]]
y = weekly["weekly_total"]

weekly


,week_start,weekly_total,week_index
0,2025-12-30,955000,0
1,2026-01-06,-187000,1
2,2026-01-13,-29000,2


Cell 6 — Train/Test Split

In [6]:
n_weeks = len(weekly)

TEST_WEEKS = 2 if n_weeks >= 4 else 1
split = max(n_weeks - TEST_WEEKS, 1)

X_train, y_train = X.iloc[:split], y.iloc[:split]
X_test, y_test   = X.iloc[split:], y.iloc[split:]

model = LinearRegression()
model.fit(X_train, y_train)

print("weeks:", n_weeks, "| test_weeks:", TEST_WEEKS)
print("coef:", model.coef_[0])
print("intercept:", model.intercept_)


weeks: 3 | test_weeks: 1
coef: -1141999.9999999998
intercept: 954999.9999999999


Cell 7 — Evaluate

In [8]:
pred_test = model.predict(X_test)

mae = mean_absolute_error(y_test, pred_test)

mse = mean_squared_error(y_test, pred_test)
rmse = np.sqrt(mse)

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))

eval_table = weekly.iloc[split:].copy()
eval_table["pred"] = pred_test
eval_table


MAE : 1300000.0
RMSE: 1300000.0


,week_start,weekly_total,week_index,pred
2,2026-01-13,-29000,2,-1329000.0


Cell 8 — Forecast Next Week

In [9]:
next_week_index = int(weekly["week_index"].max() + 1)
next_week_pred = float(model.predict(pd.DataFrame({"week_index": [next_week_index]}))[0])

last_week_start = weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(days=7)

print("Next week start:", next_week_start.date())
print("Forecast weekly_total:", round(next_week_pred, 2))


Next week start: 2026-01-20
Forecast weekly_total: -2471000.0


Cell 9 — Save Output

In [10]:
out = pd.DataFrame({
    "next_week_start": [next_week_start.date().isoformat()],
    "forecast_weekly_total": [round(next_week_pred, 2)]
})

OUT_PATH = "../reports/forecast_next_week.csv"
out.to_csv(OUT_PATH, index=False)

out
print("Saved:", OUT_PATH)


Saved: ../reports/forecast_next_week.csv
